In [1]:


import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import joblib
from pathlib import Path
import json

# Disease list
DISEASE_LIST = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE',
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 'ATRIAL_FIBRILLATION',
    'CORONARY_ARTERY_DISEASE', 'ANEMIA', 'PANCREATITIS'
]

print("="*70)
print("🔧 CLEAN FUSION MODEL RETRAINING")
print("="*70)
print(f"\nRetraining {len(DISEASE_LIST)} fusion models")
print("Goal: Restore 0.892 AUC performance\n")

🔧 CLEAN FUSION MODEL RETRAINING

Retraining 9 fusion models
Goal: Restore 0.892 AUC performance



In [2]:


print("📂 Loading data...")

# Load fusion data
X_fusion = np.load('../../data/processed/X_fusion_val.npy')
print(f"✅ X_fusion: {X_fusion.shape}")

# Load labels
y_labels = {}
for disease in DISEASE_LIST:
    disease_filename = disease.lower()
    y_labels[disease] = np.load(f'../../data/processed/y_fusion_val_{disease_filename}.npy')
print(f"✅ Labels loaded for {len(DISEASE_LIST)} diseases")

# Create EXACT same split as original
indices = np.arange(len(X_fusion))
train_val_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=42)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.20/(0.50+0.20), random_state=42)

print(f"\n📊 Split:")
print(f"   Train: {len(train_idx)} ({len(train_idx)/len(X_fusion)*100:.1f}%)")
print(f"   Val:   {len(val_idx)} ({len(val_idx)/len(X_fusion)*100:.1f}%)")
print(f"   Test:  {len(test_idx)} ({len(test_idx)/len(X_fusion)*100:.1f}%)")

📂 Loading data...
✅ X_fusion: (4643, 33)
✅ Labels loaded for 9 diseases

📊 Split:
   Train: 2321 (50.0%)
   Val:   929 (20.0%)
   Test:  1393 (30.0%)


In [3]:


def create_enhanced_features(X_base):
    """Create 105 features (33 original + 72 engineered)"""
    n_samples = X_base.shape[0]
    X_enhanced = np.zeros((n_samples, 105))
    X_enhanced[:, 0:33] = X_base
    
    agent1 = X_base[:, 0:9]
    agent2 = X_base[:, 9:18]
    agent3 = X_base[:, 18:27]
    
    idx = 33
    
    # Products (27)
    X_enhanced[:, idx:idx+9] = agent1 * agent2; idx += 9
    X_enhanced[:, idx:idx+9] = agent1 * agent3; idx += 9
    X_enhanced[:, idx:idx+9] = agent2 * agent3; idx += 9
    
    # Differences (27)
    X_enhanced[:, idx:idx+9] = np.abs(agent1 - agent2); idx += 9
    X_enhanced[:, idx:idx+9] = np.abs(agent1 - agent3); idx += 9
    X_enhanced[:, idx:idx+9] = np.abs(agent2 - agent3); idx += 9
    
    # Agreement (9)
    agreement = ((agent1 > 0.5) == (agent2 > 0.5)) & ((agent2 > 0.5) == (agent3 > 0.5))
    X_enhanced[:, idx:idx+9] = agreement.astype(float); idx += 9
    
    # Variance (9)
    all_preds = np.stack([agent1, agent2, agent3], axis=0)
    X_enhanced[:, idx:idx+9] = np.var(all_preds, axis=0)
    
    return X_enhanced

print("🔧 Creating enhanced features...")
X_enhanced = create_enhanced_features(X_fusion)
print(f"✅ Enhanced features: {X_enhanced.shape}")

# Create splits
X_train = X_enhanced[train_idx]
X_val = X_enhanced[val_idx]
X_test = X_enhanced[test_idx]

print(f"   Train: {X_train.shape}")
print(f"   Val:   {X_val.shape}")
print(f"   Test:  {X_test.shape}")

🔧 Creating enhanced features...
✅ Enhanced features: (4643, 105)
   Train: (2321, 105)
   Val:   (929, 105)
   Test:  (1393, 105)


In [4]:


print("\n" + "="*70)
print("🔨 TRAINING 9 FUSION MODELS")
print("="*70)

models = {}
val_results = {}

for disease in DISEASE_LIST:
    print(f"\n{'='*70}")
    print(f"Training: {disease}")
    print(f"{'='*70}")
    
    # Get labels
    y_train = y_labels[disease][train_idx]
    y_val = y_labels[disease][val_idx]
    y_test = y_labels[disease][test_idx]
    
    # Calculate class weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    # Train model
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc',
        early_stopping_rounds=10
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Evaluate on validation
    pred_val = model.predict_proba(X_val)[:, 1]
    auc_val = roc_auc_score(y_val, pred_val)
    
    # Evaluate on test
    pred_test = model.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, pred_test)
    
    print(f"  Validation AUC: {auc_val:.3f}")
    print(f"  Test AUC:       {auc_test:.3f}")
    
    # Store
    models[disease] = model
    val_results[disease] = {
        'val_auc': auc_val,
        'test_auc': auc_test
    }

print("\n" + "="*70)
print("✅ ALL MODELS TRAINED")
print("="*70)

# Show summary
print("\n📊 Training Summary:")
print(f"{'Disease':30s} {'Val AUC':>10s} {'Test AUC':>10s}")
print("-" * 52)
for disease in DISEASE_LIST:
    val_auc = val_results[disease]['val_auc']
    test_auc = val_results[disease]['test_auc']
    print(f"{disease:30s} {val_auc:>10.3f} {test_auc:>10.3f}")

avg_test = np.mean([val_results[d]['test_auc'] for d in DISEASE_LIST])
print("-" * 52)
print(f"{'AVERAGE':30s} {'':>10s} {avg_test:>10.3f}")
print(f"\nExpected: ~0.892")
print(f"Match: {'✅ YES!' if abs(avg_test - 0.892) < 0.02 else '⚠️ Close enough' if abs(avg_test - 0.892) < 0.05 else '❌ NO'}")


🔨 TRAINING 9 FUSION MODELS

Training: SEPSIS
  Validation AUC: 0.910
  Test AUC:       0.901

Training: PNEUMONIA
  Validation AUC: 0.858
  Test AUC:       0.886

Training: RESPIRATORY_FAILURE
  Validation AUC: 0.892
  Test AUC:       0.890

Training: ACUTE_KIDNEY_INJURY
  Validation AUC: 0.962
  Test AUC:       0.937

Training: HEART_FAILURE
  Validation AUC: 0.905
  Test AUC:       0.903

Training: ATRIAL_FIBRILLATION
  Validation AUC: 0.915
  Test AUC:       0.882

Training: CORONARY_ARTERY_DISEASE
  Validation AUC: 0.905
  Test AUC:       0.919

Training: ANEMIA
  Validation AUC: 0.838
  Test AUC:       0.837

Training: PANCREATITIS
  Validation AUC: 0.830
  Test AUC:       0.818

✅ ALL MODELS TRAINED

📊 Training Summary:
Disease                           Val AUC   Test AUC
----------------------------------------------------
SEPSIS                              0.910      0.901
PNEUMONIA                           0.858      0.886
RESPIRATORY_FAILURE                 0.892      0.89

In [5]:


print("\n" + "="*70)
print("💾 SAVING MODELS")
print("="*70)

models_dir = Path('../../models/fusion')
models_dir.mkdir(parents=True, exist_ok=True)

for disease in DISEASE_LIST:
    model_path = models_dir / f'fusion_{disease.lower()}.joblib'
    joblib.dump(models[disease], model_path)
    print(f"✅ Saved: {disease}")

print(f"\n✅ All 9 models saved to: {models_dir}")


💾 SAVING MODELS
✅ Saved: SEPSIS
✅ Saved: PNEUMONIA
✅ Saved: RESPIRATORY_FAILURE
✅ Saved: ACUTE_KIDNEY_INJURY
✅ Saved: HEART_FAILURE
✅ Saved: ATRIAL_FIBRILLATION
✅ Saved: CORONARY_ARTERY_DISEASE
✅ Saved: ANEMIA
✅ Saved: PANCREATITIS

✅ All 9 models saved to: ..\models\fusion


In [6]:


print("\n" + "="*70)
print("🔍 VERIFICATION: Load and Test Models")
print("="*70)

verification_results = []

for disease in DISEASE_LIST:
    # Load the saved model
    model_path = models_dir / f'fusion_{disease.lower()}.joblib'
    model_loaded = joblib.load(model_path)
    
    # Test on test set
    y_test = y_labels[disease][test_idx]
    pred_test = model_loaded.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, pred_test)
    
    # Compare to in-memory model
    original_auc = val_results[disease]['test_auc']
    match = abs(auc_test - original_auc) < 0.001
    
    verification_results.append({
        'Disease': disease,
        'Original AUC': original_auc,
        'Loaded AUC': auc_test,
        'Match': '✅' if match else '❌'
    })
    
    print(f"{disease:30s} Original: {original_auc:.3f}, Loaded: {auc_test:.3f} {'✅' if match else '❌'}")

verification_df = pd.DataFrame(verification_results)

print("\n" + "="*70)
all_match = all(r['Match'] == '✅' for r in verification_results)
if all_match:
    print("✅ SUCCESS! All models save/load correctly")
    print(f"✅ Average AUC: {verification_df['Loaded AUC'].mean():.3f}")
    print("\n🎉 Models are ready for:")
    print("   • Ablation study")
    print("   • Conversational interface")
    print("   • Thesis defense")
else:
    print("❌ Some models don't match - check for issues")


🔍 VERIFICATION: Load and Test Models
SEPSIS                         Original: 0.901, Loaded: 0.901 ✅
PNEUMONIA                      Original: 0.886, Loaded: 0.886 ✅
RESPIRATORY_FAILURE            Original: 0.890, Loaded: 0.890 ✅
ACUTE_KIDNEY_INJURY            Original: 0.937, Loaded: 0.937 ✅
HEART_FAILURE                  Original: 0.903, Loaded: 0.903 ✅
ATRIAL_FIBRILLATION            Original: 0.882, Loaded: 0.882 ✅
CORONARY_ARTERY_DISEASE        Original: 0.919, Loaded: 0.919 ✅
ANEMIA                         Original: 0.837, Loaded: 0.837 ✅
PANCREATITIS                   Original: 0.818, Loaded: 0.818 ✅

✅ SUCCESS! All models save/load correctly
✅ Average AUC: 0.886

🎉 Models are ready for:
   • Ablation study
   • Conversational interface
   • Thesis defense


In [7]:


print("\n" + "="*70)
print("🎯 FINAL TEST WITH OPTIMIZED THRESHOLDS")
print("="*70)

# Load thresholds
with open('../../results/optimal_thresholds.json', 'r') as f:
    thresholds = json.load(f)

final_results = []

for disease in DISEASE_LIST:
    # Load model
    model_path = models_dir / f'fusion_{disease.lower()}.joblib'
    model = joblib.load(model_path)
    
    # Get test data
    y_test = y_labels[disease][test_idx]
    pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Get threshold
    threshold = thresholds.get(disease, 0.5)
    pred_binary = (pred_proba >= threshold).astype(int)
    
    # Calculate metrics
    auc = roc_auc_score(y_test, pred_proba)
    f1 = f1_score(y_test, pred_binary)
    precision = precision_score(y_test, pred_binary, zero_division=0)
    recall = recall_score(y_test, pred_binary, zero_division=0)
    
    final_results.append({
        'Disease': disease,
        'AUC': auc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall,
        'Threshold': threshold
    })
    
    print(f"{disease:30s} AUC: {auc:.3f}, F1: {f1:.3f} (thresh: {threshold:.2f})")

final_df = pd.DataFrame(final_results)

print("\n" + "="*70)
print("📊 FINAL PERFORMANCE")
print("="*70)
print(f"   Average AUC:       {final_df['AUC'].mean():.3f}")
print(f"   Average F1:        {final_df['F1'].mean():.3f}")
print(f"   Average Precision: {final_df['Precision'].mean():.3f}")
print(f"   Average Recall:    {final_df['Recall'].mean():.3f}")
print()
print(f"   Expected AUC: ~0.892")
print(f"   Expected F1:  ~0.710")
print()
if abs(final_df['AUC'].mean() - 0.892) < 0.02:
    print("✅ MODELS SUCCESSFULLY RESTORED!")
    print("✅ Ready for ablation study and conversational interface!")
else:
    print("⚠️ Close but not exact - acceptable variance")


🎯 FINAL TEST WITH OPTIMIZED THRESHOLDS
SEPSIS                         AUC: 0.901, F1: 0.680 (thresh: 0.54)
PNEUMONIA                      AUC: 0.886, F1: 0.684 (thresh: 0.33)
RESPIRATORY_FAILURE            AUC: 0.890, F1: 0.751 (thresh: 0.56)
ACUTE_KIDNEY_INJURY            AUC: 0.937, F1: 0.842 (thresh: 0.54)
HEART_FAILURE                  AUC: 0.903, F1: 0.768 (thresh: 0.32)
ATRIAL_FIBRILLATION            AUC: 0.882, F1: 0.725 (thresh: 0.41)
CORONARY_ARTERY_DISEASE        AUC: 0.919, F1: 0.780 (thresh: 0.39)
ANEMIA                         AUC: 0.837, F1: 0.681 (thresh: 0.43)
PANCREATITIS                   AUC: 0.818, F1: 0.400 (thresh: 0.27)

📊 FINAL PERFORMANCE
   Average AUC:       0.886
   Average F1:        0.701
   Average Precision: 0.646
   Average Recall:    0.806

   Expected AUC: ~0.892
   Expected F1:  ~0.710

✅ MODELS SUCCESSFULLY RESTORED!
✅ Ready for ablation study and conversational interface!
